In [1]:
# Identifiers (RQx, Hx, Px, Gx, Fx, Tx) are defined in vpir_bench_spec.md: section 2 for the research questions,
# section 11 for the tests behind each gate, section 12 for phases, gates and figures. Rx reproduction targets are
# in docs/reproduction_targets.md, and numbered decisions in plan.md.
# RQ2: how does the verifiable/baseline overhead vary with database size and record size? (spec 2, H2)
import json, sys
from pathlib import Path

import pandas as pd

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.analysis import frame, plots
from cs4406_sem_project.harness.grid import fmt_size

RUNS = [repo / "results/raw/all-20261112_tier1", repo / "results/raw/all-20261112_anchors"]  # notebook 17
platform = json.loads((RUNS[0] / "meta.json").read_text())["platform"]
df, status = frame.load_runs(RUNS, platform)
t = plots.build(df, status)
FIG = repo / "docs/figures"
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 400)


def save(name, fig):
    for ext in ("pdf", "png"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=200, bbox_inches="tight")


def save_table(name, table):
    (FIG / f"{name}.md").write_text(plots._markdown(table, plots.TABLE_TITLES[name], False), encoding="utf-8")


plots._style()
print(f"{len(df)} measurement rows from {', '.join(r.name for r in RUNS)}; {len(status)} (cell, phase) rows not ok")

3427 measurement rows from all-20261112_tier1, all-20261112_anchors; 14 (cell, phase) rows not ok


In [2]:
# T4: overhead = median(verifiable) / median(baseline) per cell, with paired bootstrap 95 % CIs (10 000 resamples),
# for the three price pairs and the S0-go/S0-cpp implementation ablation. Cells padded above 1.1 are flagged
# (decision 19).
t4 = plots.t4_overheads(t)
save_table("T4_overheads", t4)
METRICS = list(plots.F2_METRICS)
view = t.overheads[t.overheads.metric.isin(METRICS)]
for (pair, metric), g in view.groupby(["pair", "metric"]):
    tab = g.pivot_table(index="db_bytes", columns="record_bytes", values="ratio")
    tab.index = [fmt_size(int(i)) for i in tab.index]
    tab.columns = [fmt_size(int(c)) for c in tab.columns]
    print(f"{pair:14s} {metric}\n{tab.round(3).to_string()}\n")

M1/M0          client_online_s
          32B   256B   1KiB   8KiB
1MiB    8.369  3.184  2.933  3.088
16MiB   4.588  3.784  3.378  3.016
128MiB  3.614  3.330  3.073  2.786
1GiB      NaN    NaN  2.961  2.266

M1/M0          client_storage_bytes
Empty DataFrame
Columns: []
Index: []

M1/M0          online_bytes
           32B   256B   1KiB   8KiB
1MiB    16.159  2.534  1.321  1.028
16MiB   20.131  3.034  1.446  1.044
128MiB  23.194  3.408  1.540  1.056
1GiB       NaN    NaN  1.634  1.068

M1/M0          server_online_s
           32B   256B   1KiB   8KiB
1MiB    11.057  2.114  1.201  0.980
16MiB   12.609  2.492  1.360  0.998
128MiB  14.140  2.761  1.478  1.025
1GiB       NaN    NaN  1.554  1.016

S0-go/S0-cpp   client_online_s
          32B   256B   1KiB   8KiB
1MiB    0.495  0.497  0.414  0.101
16MiB   0.486  0.484  0.508  0.262
128MiB  0.491  0.492  0.483  0.313
1GiB    0.519  0.523  0.514  0.521

S0-go/S0-cpp   client_storage_bytes
          32B   256B   1KiB   8KiB
1MiB    0.404  0.40

In [3]:
# H2: the overhead is largest for small records and small databases, and shrinks with database size. Per pair and
# metric: is the largest ratio at the smallest (database, record) cell measured, and does the ratio fall from the
# smallest to the largest database at each record size?
verdicts = []
for (pair, metric), g in view.groupby(["pair", "metric"]):
    g = g.dropna(subset=["ratio"])
    if g.empty:
        continue
    top = g.loc[g.ratio.idxmax()]
    smallest = (g.db_bytes.min(), g.record_bytes.min())
    falls = [g[g.record_bytes == r].sort_values("db_bytes").ratio.iloc[-1] < g[g.record_bytes == r].sort_values("db_bytes").ratio.iloc[0]
             for r in sorted(g.record_bytes.unique()) if (g.record_bytes == r).sum() > 1]
    verdicts.append({"pair": pair, "metric": metric, "max_ratio": round(top.ratio, 3),
                     "max_at": f"{fmt_size(int(top.db_bytes))} x {fmt_size(int(top.record_bytes))}",
                     "max_at_smallest_cell": (top.db_bytes, top.record_bytes) == smallest,
                     "falls_with_db_at": f"{sum(falls)} of {len(falls)} record sizes"})
print(pd.DataFrame(verdicts).to_string(index=False))

        pair               metric  max_ratio       max_at  max_at_smallest_cell    falls_with_db_at
       M1/M0      client_online_s      8.369   1MiB x 32B                  True 2 of 4 record sizes
       M1/M0         online_bytes     23.194 128MiB x 32B                 False 0 of 4 record sizes
       M1/M0      server_online_s     14.140 128MiB x 32B                 False 0 of 4 record sizes
S0-go/S0-cpp      client_online_s      0.523  1GiB x 256B                 False 0 of 4 record sizes
S0-go/S0-cpp client_storage_bytes      0.422 16MiB x 8KiB                 False 1 of 4 record sizes
S0-go/S0-cpp         online_bytes      0.845 16MiB x 8KiB                 False 1 of 4 record sizes
S0-go/S0-cpp      server_online_s      0.869 16MiB x 256B                 False 0 of 4 record sizes
   S1/S0-cpp      client_online_s      1.066   1MiB x 32B                  True 1 of 4 record sizes
   S1/S0-cpp client_storage_bytes      1.011 128MiB x 32B                 False 0 of 4 record sizes


In [4]:
# F2 (overhead heatmaps), F3 (absolute cost against size, S4 extrapolated past 1 MiB) and F6 (where the
# verification cost goes, at Q = 100).
for name in ("F2_overhead_heatmaps", "F3_absolute_cost", "F6_breakdown"):
    save(name, plots.FIGURE_BUILDERS[name](t))
print(sorted(p.name for p in FIG.glob("F[236]_*")))
print("S4 fit exponents:", t.fits.set_index("metric").exponent.round(3).to_dict())

['F2_overhead_heatmaps.pdf', 'F2_overhead_heatmaps.png', 'F3_absolute_cost.pdf', 'F3_absolute_cost.png', 'F6_breakdown.pdf', 'F6_breakdown.png']
S4 fit exponents: {'client_offline_s': 0.471, 'client_online_s': 0.492, 'client_recover_s': 0.497, 'client_storage_bytes': 0.499, 'down_bytes': 0.5, 'offline_bytes': 0.499, 'offline_down_bytes': 0.499, 'offline_down_bytes_hint_only': 0.499, 'online_bytes': 0.5, 'query_gen_s': 0.49, 'server_offline_s': 0.984, 'server_online_s': 0.986, 'server_preproc_s': 0.984, 'throughput_GBps': 0.014, 'up_bytes': 0.5, 'user_time_s': 0.956}
